In [ ]:
import math
import random

random.seed(23)

def euclidean_distance(a: tuple, b: tuple) -> float:
    """Computes the euclidean distance.

    Args:
        a (tuple): _description_
        b (tuple): _description_

    Returns:
        float: _description_
    """
    
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def tour_length(tour, dists):
    """Computes the length of a tour.

    Args:
        tour (_type_): _description_
        dists (_type_): _description_

    Returns:
        _type_: _description_
    """
    n = len(tour)
    total = 0
    for k in range(n - 1):
        total += dists[tour[k]][tour[k+1]]
    total += dists[tour[-1]][tour[0]]
    return total

In [95]:
import math
import random

def load_tsp(filename: str) -> list:
    """Function that loads data from the .tsp files based on the formatation prerequisites

    Args:
        filename (str): _description_

    Returns:
        list: _description_
    """
    coords = []
    with open(filename) as f:
        for line in f:
            if line.strip() == 'NODE_COORD_SECTION':
                break
        for line in f:
            if line.strip() == 'EOF':
                break
            parts = line.split()
            x, y = float(parts[1]), float(parts[2])
            coords.append((x, y))
            
    
    n = len(coords)
    
    dists = [[euclidean_distance(coords[i], coords[j]) 
         for j in range(n)] 
         for i in range(n)]
    
    return coords, dists

In [ ]:
def order_crossover(parent1, parent2, crossover_rate):
    """Creates a crossover using the order method.

    Args:
        parent1 (_type_): _description_
        parent2 (_type_): _description_
        crossover_rate (_type_): _description_

    Returns:
        _type_: _description_
    """
    n = len(parent1)
    child1, child2 = list(parent1), list(parent2)

    if random.random() < crossover_rate:
        i = random.randint(0, n - 2)
        j = random.randint(i + 1, n - 1)
        
        subtour1 = parent1[i:j+1]
        sub_set1 = set(subtour1) 
        
        order_p2 = parent2[j+1:] + parent2[:j+1]
        remaining1 = [item for item in order_p2 if item not in sub_set1]
        
        c1 = [None] * n
        c1[i:j+1] = subtour1
        c1[j+1:] = remaining1[:n-(j+1)]
        c1[:i] = remaining1[n-(j+1):]
        child1 = c1

    if random.random() < crossover_rate:
        i = random.randint(0, n - 2)
        j = random.randint(i + 1, n - 1)
        
        subtour2 = parent2[i:j+1]
        sub_set2 = set(subtour2)
        
        order_p1 = parent1[j+1:] + parent1[:j+1]
        remaining2 = [item for item in order_p1 if item not in sub_set2]
        
        c2 = [None] * n
        c2[i:j+1] = subtour2
        c2[j+1:] = remaining2[:n-(j+1)]
        c2[:i] = remaining2[n-(j+1):]
        child2 = c2
            
    return child1, child2

In [ ]:
import random

def pmx_crossover(parent1, parent2, crossover_rate):
    """Creates a crossover using the pmx version.

    Args:
        parent1 (_type_): _description_
        parent2 (_type_): _description_
        crossover_rate (_type_): _description_

    Returns:
        _type_: _description_
    """
    n = len(parent1)
    
    child1, child2 = list(parent1), list(parent2)

    if random.random() < crossover_rate:
        i = random.randint(0, n - 2)
        j = random.randint(i + 1, n - 1)
        
        c1 = [None] * n
        c1[i:j+1] = parent1[i:j+1]
        
        for k in range(i, j + 1):
            val_p2 = parent2[k]
            if val_p2 not in c1:
                val_p1 = parent1[k]
                target_idx = parent2.index(val_p1)
                
                while i <= target_idx <= j:
                    val_at_target_in_p1 = parent1[target_idx]
                    target_idx = parent2.index(val_at_target_in_p1)
                
                c1[target_idx] = val_p2
        
        for k in range(n):
            if c1[k] is None:
                c1[k] = parent2[k]
        child1 = c1

    if random.random() < crossover_rate:
        i = random.randint(0, n - 2)
        j = random.randint(i + 1, n - 1)
        
        c2 = [None] * n
        c2[i:j+1] = parent2[i:j+1]
        
        for k in range(i, j + 1):
            val_p1 = parent1[k]
            if val_p1 not in c2:
                val_p2 = parent2[k]
                target_idx = parent1.index(val_p2)
                
                while i <= target_idx <= j:
                    val_at_target_in_p2 = parent2[target_idx]
                    target_idx = parent1.index(val_at_target_in_p2)
                
                c2[target_idx] = val_p1
                
        for k in range(n):
            if c2[k] is None:
                c2[k] = parent1[k]
        child2 = c2
            
    return child1, child2

In [ ]:
# inversion mutation
def mutate(individual) :
    """Mutates instances of the tsp solution.

    Args:
        individual (_type_): _description_

    Returns:
        _type_: _description_
    """
    
    mutation_rate = 0.8
    
    if random.random() < mutation_rate :
    
        n = len(individual)
        
        i = random.randint(0, n - 2)
        j = random.randint(i + 1, n - 1)
        individual[i:j+1] = individual[i:j+1][::-1]
            
    return individual

def tournament_selection(pop_with_fitness, tournament_size=3):
    """The function that selects parents .

    Args:
        pop_with_fitness (_type_): _description_
        tournament_size (int, optional): _description_. Defaults to 3.

    Returns:
        _type_: _description_
    """
    candidates = random.sample(pop_with_fitness, tournament_size)
    return min(candidates, key=lambda x: x[1])[0]
            

In [ ]:
def greedy_tour_from(start, coords, dists):
    """Function that computes the greedy tour.

    Args:
        start (_type_): _description_
        coords (_type_): _description_
        dists (_type_): _description_

    Returns:
        _type_: _description_
    """
    
    n = len(coords)
    tour = [start]
    available = set(range(n))
    available.remove(start)
    current = start

    while available:
        nearest = min(available, key=lambda node: dists[current][node])
        tour.append(nearest)
        available.remove(nearest)
        current = nearest

    return tour

In [ ]:
def evolutionary_algorithm_tsp(filename, iterations, initial_population, elite_size, crossover_type):
    """An implementation of an EA algorithm for the tsp problem.

    Args:
        filename (_type_): _description_
        iterations (_type_): _description_
        initial_population (_type_): _description_
        elite_size (_type_): _description_
        crossover_type (_type_): _description_

    Returns:
        _type_: _description_
    """
    coords, dists = load_tsp(filename)
    n = len(coords)
    
    population = []
    
    for i in range(initial_population//2):
        greedy_tour = greedy_tour_from(i%n, coords, dists)
        population.append(greedy_tour)
        
    while len(population) < initial_population :
        random_entity = list(range(n))
        random.shuffle(random_entity)
        population.append(random_entity)
    
    best_ever = float('inf')
    no_improve_count = 0
    no_improve_limit = 200  
    
    for gen in range(iterations):
        pop_with_fitness = [(ind, tour_length(ind, dists)) for ind in population]
        pop_with_fitness.sort(key=lambda x: x[1])
        
        new_population = [ind for ind, fit in pop_with_fitness[:elite_size]]
        
        current_best = pop_with_fitness[0][1]
        if current_best < best_ever:
            best_ever = current_best
            no_improve_count = 0
        else:
            no_improve_count += 1
        
        if no_improve_count >= no_improve_limit:
            for _ in range(20):
                entity = list(range(n))
                random.shuffle(entity)
                new_population.append(entity)
            no_improve_count = 0
        
        while len(new_population) < initial_population:
            p1 = tournament_selection(pop_with_fitness)
            p2 = tournament_selection(pop_with_fitness)
            
            if crossover_type == "order" :
                c1, c2 = order_crossover(p1, p2, 0.7)
            if crossover_type == "pmx" :
                c1, c2 = pmx_crossover(p1, p2, 0.7)
            
            c1 = mutate(c1)
            new_population.append(c1)
            
            if len(new_population) < initial_population:
                c2 = mutate(c2)
                new_population.append(c2)
        
        population = new_population
    
    final = [(ind, tour_length(ind, dists)) for ind in population]
    final.sort(key=lambda x: x[1])
    return final[0][1]

In [103]:
# 26051. for Rwanda
# 7542 for Berlin
evolutionary_algorithm_tsp('../Lab3Assigment3/berlin52.tsp', 5000, 100, 10)

Gen 0: Best = 8182.1915557256725
Gen 10: Best = 8062.031482471483
Gen 20: Best = 7878.692733458706
Gen 30: Best = 7850.7674115679465
Gen 40: Best = 7796.564198460165
Gen 50: Best = 7742.802280698529
Gen 60: Best = 7742.802280698529
Gen 70: Best = 7742.802280698529
Gen 80: Best = 7742.802280698529
Gen 90: Best = 7742.802280698529
Gen 100: Best = 7721.29791869682


KeyboardInterrupt: 

In [104]:
evolutionary_algorithm_tsp('../Lab3Assigment3/rw1621.tsp', 500, 20, 10)

KeyboardInterrupt: 

In [ ]:
def generate_report(filename: str, output: str,
                    iterations_values, population_values,
                    elite_values, crossover_types):
    """The function that generates the reports.

    Args:
        filename (str): _description_
        output (str): _description_
        iterations_values (_type_): _description_
        population_values (_type_): _description_
        elite_values (_type_): _description_
        crossover_types (_type_): _description_
    """

    with open(output, 'a') as f:
        f.write(f"\nReport for {filename}\n")

    for crossover_type in crossover_types:

        with open(output, 'a') as f:
            f.write(f"\nChecking for {crossover_type} crossover different values\n")

        for iterations in iterations_values:
            for population in population_values:
                for elite in elite_values:

                    result = evolutionary_algorithm_tsp(filename, 
                                                        iterations=iterations,
                                                        initial_population=population,
                                                        elite_size=elite,
                                                        crossover_type=crossover_type)

                    with open(output, 'a') as f:
                        f.write(f"iterations={iterations}, population={population}, elite={elite}, crossover={crossover_type} => result={result:.2f}\n")

In [113]:
generate_report('../Lab3Assigment3/berlin52.tsp', 'report_ea_berlin52.txt',
                iterations_values=[100, 500, 1000, 2500],
                population_values=[20, 50, 100],
                elite_values=[3, 5, 10],
                crossover_types=['order', 'pmx'])

In [114]:
generate_report('../Lab3Assigment3/rw1621.tsp', 'report_ea_rw1621.txt',
                iterations_values=[50, 100, 200],
                population_values=[10, 20, 30],
                elite_values=[3, 5, 10],
                crossover_types=['order', 'pmx'])